In [1]:
import pandas as pd

In [6]:
city = pd.read_excel("Tourism Dataset/City.xlsx")
continent = pd.read_excel("Tourism Dataset/Continent.xlsx")
country = pd.read_excel("Tourism Dataset/Country.xlsx")
item = pd.read_excel("Tourism Dataset/Updated_Item.xlsx")
mode = pd.read_excel("Tourism Dataset/Mode.xlsx")
region = pd.read_excel("Tourism Dataset/Region.xlsx")
transaction = pd.read_excel("Tourism Dataset/Transaction.xlsx")
types = pd.read_excel("Tourism Dataset/Type.xlsx")
users = pd.read_excel("Tourism Dataset/User.xlsx")

In [7]:
transaction.head()

,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating
0,3,70456,2022,10,2,640,5
1,8,7567,2022,10,4,640,5
2,9,79069,2022,10,3,640,5
3,10,31019,2022,10,3,640,3
4,15,43611,2022,10,2,640,3


In [8]:
df = transaction.merge(users, on='UserId', how='left')
df = df.merge(item, on='AttractionId', how='left')
df = df.merge(types, on='AttractionTypeId', how='left')

In [9]:
user_city = city.rename(columns={
    'CityId': 'UserCityId',
    'CityName': 'UserCity'
})

attraction_city = city.rename(columns={
    'CityId': 'AttractionCityId',
    'CityName': 'AttractionCity'
})

In [10]:
df = df.merge(
    user_city[['UserCityId', 'UserCity']],
    left_on='CityId',
    right_on='UserCityId',
    how='left'
)

df = df.merge(
    attraction_city[['AttractionCityId', 'AttractionCity']],
    on='AttractionCityId',
    how='left'
)

In [11]:
df = df.merge(
    country[['CountryId', 'Country']],
    on='CountryId',
    how='left'
)

df = df.merge(
    region[['RegionId', 'Region']],
    on='RegionId',
    how='left'
)

df = df.merge(
    continent[['ContinentId', 'Continent']],
    on='ContinentId',
    how='left'
)

In [12]:
df.columns

Index(['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitMode',
       'AttractionId', 'Rating', 'ContinentId', 'RegionId', 'CountryId',
       'CityId', 'AttractionCityId', 'AttractionTypeId', 'Attraction',
       'AttractionAddress', 'AttractionType', 'UserCityId', 'UserCity',
       'AttractionCity', 'Country', 'Region', 'Continent'],
      dtype='str')

In [13]:
duplicate_cols = [col for col in df.columns if '_x' in col or '_y' in col]

df.drop(columns=duplicate_cols, inplace=True)

In [14]:
df['combined_text'] = (
    df['Attraction'].astype(str) + ' ' +
    df['AttractionAddress'].astype(str)
)

In [15]:
final_columns = [
    'TransactionId',
    'UserId',
    'VisitYear',
    'VisitMonth',
    'VisitMode',
    'Rating',
    'AttractionId',
    'Attraction',
    'AttractionAddress',
    'AttractionType',
    'UserCity',
    'AttractionCity',
    'Country',
    'Region',
    'Continent',
    'combined_text'
]

final_df = df[final_columns]

In [16]:
final_df.shape

(52930, 16)

In [17]:
final_df.isnull().sum()

TransactionId        0
UserId               0
VisitYear            0
VisitMonth           0
VisitMode            0
Rating               0
AttractionId         0
Attraction           0
AttractionAddress    0
AttractionType       0
UserCity             8
AttractionCity       0
Country              0
Region               0
Continent            0
combined_text        0
dtype: int64

In [18]:
final_df.head()

,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,Rating,AttractionId,Attraction,AttractionAddress,AttractionType,UserCity,AttractionCity,Country,Region,Continent,combined_text
0,3,70456,2022,10,2,5,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Guildford,Douala,United Kingdom,Western Europe,Europe,Sacred Monkey Forest Sanctuary Jl. Monkey Fore...
1,8,7567,2022,10,4,5,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Ontario,Douala,Canada,Northern America,America,Sacred Monkey Forest Sanctuary Jl. Monkey Fore...
2,9,79069,2022,10,3,5,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Brazil,Douala,Brazil,South America,America,Sacred Monkey Forest Sanctuary Jl. Monkey Fore...
3,10,31019,2022,10,3,3,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Zurich,Douala,Switzerland,Central Europe,Europe,Sacred Monkey Forest Sanctuary Jl. Monkey Fore...
4,15,43611,2022,10,2,3,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Manchester,Douala,United Kingdom,Western Europe,Europe,Sacred Monkey Forest Sanctuary Jl. Monkey Fore...


In [19]:
final_df.to_csv("tourism_cleaned.csv", index=False)